# Can I run these?

*Checks your credentials for real, then tells you which notebook in this repo you can run right now.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/00_check_setup.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** Every notebook here declares what it needs. This one checks what you have and joins the two.
**Result.** A per-notebook verdict: ready, or blocked and on what.
**Requires.** nothing
**Run** ~1 min · **Cost** ~$0.00 · no stored output — run it to see anything

Start here. Nothing below sets anything up — it only looks — so it is safe to run
repeatedly, and it is the fastest way to find out whether a missing key or a paused
cluster is about to waste twenty minutes of your time.

**This is the one notebook here with no stored output.** Every other notebook commits its
results so you can read it on GitHub without running it. This one's results would only
describe whoever ran it last — their cluster, their provider, their machine — so it ships
empty on purpose. It has nothing to tell you until you run it.

The checks are **live**: a real connection to Couchbase, a real call to your model
provider. Confirming that a key is *present* is worthless — a revoked key looks
identical to a working one until something tries to use it.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

# This notebook deliberately requires nothing: it has to run when nothing works.
settings = cbnb.bootstrap(requires=[])

## 1. What this environment can do

Each row is one **capability** — a coarse, named thing a notebook can ask for. The
checks below actually exercise them, so this takes a few seconds and may download the
embedding model the first time.

In [ ]:
from cbnb import readiness
from cbnb.readout import Item, Panel

results = readiness.report()  # live checks: real connection, real API call

working = sum(r.ok for r in results)
Panel(
    f"{working} of {len(results)} capabilities working",
    [("Capabilities", [Item("ok" if r.ok else "blocked", r.name, r.detail, r.fix)
                       for r in results])],
    status="ok" if working == len(results) else "blocked",
)

## 2. Which notebooks you can run

Every notebook states its requirements in its own title cell and setup call. This reads
those declarations straight from the files — nothing is registered anywhere, so a
notebook added tomorrow appears here without this one being edited.

In [ ]:
from collections import Counter

from cbnb import inventory

have = {r.name for r in results if r.ok}
fix_for = {r.name: r.fix for r in results}

ready, blocked, undeclared = [], [], []
unlocks = Counter()  # capability -> notebooks it is the *only* thing blocking

for lab in inventory.labs():
    if lab.name.startswith("00_"):
        continue  # this notebook
    where = f"{lab.track}/{lab.name}" if lab.track else lab.name
    missing = [c for c in lab.requires if c not in have]
    if not lab.declared_in:
        undeclared.append(Item("unknown", lab.title, where, "requirements not declared"))
    elif missing:
        blocked.append(Item("blocked", lab.title, f"{where} · needs {', '.join(missing)}"))
        if len(missing) == 1:
            unlocks[missing[0]] += 1
    else:
        ready.append(Item("ok", lab.title, where))

if unlocks:
    best, count = unlocks.most_common(1)[0]
    more = "notebook" if count == 1 else "notebooks"
    summary = f"Fix {best} first: that alone unlocks {count} more {more}. Its fix is above."
elif blocked:
    summary = "Each blocked notebook needs more than one fix; see the capabilities above."
else:
    summary = "Nothing is blocked."

Panel(
    f"You can run {len(ready)} of {len(ready) + len(blocked) + len(undeclared)} notebooks",
    [("Ready", ready), ("Blocked", blocked), ("Not declared", undeclared)],
    summary=summary,
    status="ok" if not blocked and not undeclared else "blocked",
)

## 3. If something came back blocked

**A missing setting.** The note under each blocked capability is specific to where you are running —
`.env` locally, the secrets panel on Colab, repository secrets on Codespaces. The names
themselves are all in [`.env.example`](../.env.example).

**A rejected key.** The provider answered and said no. That is a wrong, expired or
revoked key rather than a missing one, and re-pasting the same value will not help — check
it in your provider's dashboard first. To try a different one for the rest of this session
without editing any file:

```python
cbnb.update_setting("NANOGPT_API_KEY")   # or OPENAI_API_KEY, GROQ_API_KEY, ...
```

Then re-run the check above. That only lasts for the session; to keep it, put it where the
note says.

**Couchbase unreachable.** Almost always one of three things: the cluster is paused, your
IP is not on its allow list, or you are using your Capella *login* rather than a database
access user. [`docs/capella-setup.md`](../docs/capella-setup.md) walks through all three.

**Nothing blocked?** Start with
[`retrieval/01_building_hybrid_search.ipynb`](retrieval/01_building_hybrid_search.ipynb);
the [README](../README.md) lists the rest in reading order.